# IoMT-Warden: Cyber-Physical Security for IoMT with Safe Reinforcement Learning & Digital Twin

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anzar0904/iot-reinforcement-learning/blob/main/IoMT_Warden_Colab.ipynb)

This Google Colab notebook is an interactive walkthrough and test harness for **IoMT-Warden**, an autonomous multi-layer defense system for Internet of Medical Things (IoMT) devices.

### 5-Pillar Architecture:
1. **Synthetic Telemetry Simulation** (Healthy, Arrhythmia, Alzheimer patient models)
2. **Patient Digital Twin** (Real-time EWMA state estimation & MAD residual tracking)
3. **Cryptographic Gateway** (HMAC-SHA256 integrity, replay prevention, and nonce deduplication)
4. **Machine Learning Anomaly Detection** (Quantile-calibrated Isolation Forest)
5. **Safe Deep Q-Network (DQN)** (Deep RL defense agent with an override-proof clinical safety shield)

## 1. Clone Repository & Setup Environment

In [ ]:
# Clone the repository and install required packages
!git clone https://github.com/Anzar0904/iot-reinforcement-learning.git
%cd iot-reinforcement-learning
!pip install -q -r iot_reinforcement_learning/requirements.txt
!pip install -q gymnasium torch scikit-learn pandas matplotlib

## 2. Verify Core Defense Engines
Test imports and verify all mathematical and reinforcement learning components.

In [ ]:
import sys
sys.path.append(".")

from iot_reinforcement_learning.core import PatientDigitalTwin, CryptoEngine, IoMTSimulator
from iot_reinforcement_learning.ml_detector import IoMTAnomalyDetector
from iot_reinforcement_learning.rl_defense import IoMTEnv, DQNAgent, SafeIoMTDefender

print("✓ All IoMT-Warden engines loaded successfully!")

## 3. Real-Time Telemetry & Digital Twin Stream

In [ ]:
# Initialize Simulator, Digital Twin, and Crypto Gateway
twin = PatientDigitalTwin(alpha=0.25, mad_window=50)
crypto = CryptoEngine(secret_key=b"colab-demo-key-1234")
sim = IoMTSimulator(patient_type="heart_issued")

print("Streaming 10 simulated IoMT telemetry packets:")
print("-" * 80)
for i in range(10):
    packet = sim.generate_packet(seq_id=i)
    signed = crypto.sign_packet(packet)
    is_valid, msg = crypto.verify_packet(signed)
    twin_res = twin.update(packet)
    hr_pred = twin_res["predicted"]["heart_rate"]
    print(f"Packet #{i:02d} | HMAC: {'VALID' if is_valid else 'INVALID'} | HR: {packet['heart_rate']:5.1f} (EWMA: {hr_pred:5.1f}) | SpO2: {packet['spo2']:5.1f}% | Anomaly: {twin_res['is_anomaly']}")
print("-" * 80)

## 4. Test Pretrained Safe DQN with Safety Shield

In [ ]:
# Load the trained 250-episode model weights
defender = SafeIoMTDefender(weights_path="weights/dqn_iomt_warden.pt")

scenarios = [
    {"name": "Normal Telemetry", "state": [0.05, 0.02, 0.0, 0.0, 0.98, 0.0, 1.0, 0.0]},
    {"name": "FDI Attack (Spoofed Biosensors)", "state": [3.8, 2.9, 1.0, 1.0, 0.15, 0.85, 0.0, 0.0]},
    {"name": "Replay / Cryptographic Attack", "state": [0.0, 0.0, 0.0, 0.0, 0.95, 0.0, 0.0, 0.0]},
    {"name": "True Cardiac Emergency", "state": [4.2, 3.1, 1.0, 1.0, 0.10, 0.92, 1.0, 1.0]},
]

print(f"{'Scenario':<32} | {'Action Taken':<16} | {'Shield Overrode DQN?'}")
print("-" * 68)
for sc in scenarios:
    action_idx, meta = defender.select_action(sc["state"])
    shield_str = "YES (Emergency Priority)" if meta["shield_applied"] else "No (Policy Allowed)"
    print(f"{sc['name']:<32} | {meta['action_name']:<16} | {shield_str}")
print("-" * 68)

## 5. Research Benchmark Evaluation (5,000 Sample Rapid Run)

In [ ]:
from experiments.run_evaluation import run_evaluation_suite
import pandas as pd

print("Evaluating Rule-Based vs Isolation Forest vs IoMT-Warden DQN (5,000 cycles)...")
results, df_samples = run_evaluation_suite(n_samples=5000)
summary_df = pd.DataFrame(results).T
print("
--- Benchmark Metrics ---")
print(summary_df[["detection_rate", "false_alarm_rate", "missed_emergencies", "mean_clinical_availability", "composite_score"]])

## 6. Publication Figures Visualizer

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

fig_paths = [
    ("Figure 1: DQN Reward Convergence Curve", "experiments/figures/reward_curve.png"),
    ("Figure 2: Digital Twin Telemetry Tracking", "experiments/figures/twin_prediction.png"),
    ("Figure 3: ML Anomaly Residual Calibration", "experiments/figures/anomaly_score.png"),
    ("Figure 4: Comparative Method Performance Benchmark", "experiments/figures/method_comparison.png")
]

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
for ax, (title, path) in zip(axes.flatten(), fig_paths):
    if os.path.exists(path):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.set_title(title, fontsize=13, fontweight="bold", pad=10)
    else:
        ax.text(0.5, 0.5, f"File not found: {path}", ha="center", va="center")
    ax.axis("off")
plt.tight_layout()
plt.show()